In [1]:
!pip install sentence-transformers gensim nltk

  Using cached smart_open-7.5.1-py3-none-any.whl.metadata (24 kB)
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   --- ------------------------------------ 1.8/24.4 MB 11.2 MB/s eta 0:00:03
   ------ --------------------------------- 4.2/24.4 MB 11.5 MB/s eta 0:00:02
   ---------- ----------------------------- 6.6/24.4 MB 10.9 MB/s eta 0:00:02
   -------------- ------------------------- 8.7/24.4 MB 11.2 MB/s eta 0:00:02
   ------------------ --------------------- 11.0/24.4 MB 10.9 MB/s eta 0:00:02
   ---------------------- ----------------- 13.6/24.4 MB 11.1 MB/s eta 0:00:01
   ------------------------- -------------- 15.7/24.4 MB 11.0 MB/s eta 0:00:01
   ----------------------------- ---------- 18.1/24.4 MB 11.1 MB/s eta 0:00:01
   --------------------------------- ------ 20.4/24.4 MB 11.0 MB/s eta 0:00:01
   ------------------------------------ --- 22.3/24.4 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------  24.1/24.4 MB 10.8 MB/s eta 0:00:01

In [2]:
import pandas as pd
import numpy as np
import re
import os
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"Device   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# ── CONFIG ────────────────────────────────────────────────
PAIRS_CSV   = "email_reply_pairs.csv"
OUTPUT_DIR  = "embeddings"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SBERT_MODEL = "all-MiniLM-L6-v2"   # 80MB, fast, great quality
W2V_DIM     = 128                   # Word2Vec vector size
W2V_WINDOW  = 5
W2V_EPOCHS  = 10
# ──────────────────────────────────────────────────────────

print("\nConfig loaded ✅")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rohru\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rohru\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rohru\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


PyTorch  : 2.7.1+cu118
CUDA     : True
Device   : NVIDIA GeForce RTX 4060 Laptop GPU

Config loaded ✅


In [3]:
df = pd.read_csv(PAIRS_CSV)

print(f"Pairs loaded    : {len(df):,}")
print(f"Columns         : {list(df.columns)}")
print(f"\nEmail wc range  : {df['email_wc'].min()} – {df['email_wc'].max()}")
print(f"Reply wc range  : {df['reply_wc'].min()} – {df['reply_wc'].max()}")
print(f"\nSample email:\n{df['email_body'].iloc[0][:300]}")
print(f"\nSample reply:\n{df['reply_body'].iloc[0][:200]}")

Pairs loaded    : 48,184
Columns         : ['email_id', 'reply_id', 'email_from', 'reply_from', 'subject', 'subject_norm', 'email_body', 'reply_body', 'email_wc', 'reply_wc']

Email wc range  : 5 – 600
Reply wc range  : 3 – 600

Sample email:
Time is running very short. Is your company prepared for Etag 1.7? These are the minimum REQUIRED steps that must be completed prior to March 6, 2002:

---------------------------

(1) All entities must update the NERC Registry at http://www.tsin.com.

(1a) CA's and TP's must designate PORs and PODs

Sample reply:
Cara,

Do you know about this and are we ready for this version? 

Caroline


In [4]:
STOPWORDS = set(stopwords.words('english'))

# extra email-specific noise words
EMAIL_NOISE = {
    'enron','ect','hou','corp','company','email','message',
    'mailto','http','www','com','cc','bcc','subject','original',
    'forwarded','sent','received','dear','regards','thanks',
    'thank','please','would','could','also','may','one','two',
    'per','get','let','know','hi','hello','yes','no'
}
ALL_STOPWORDS = STOPWORDS | EMAIL_NOISE

def preprocess(text: str) -> list[str]:
    """
    Lowercase → strip URLs/emails/numbers 
    → tokenize → remove stopwords + short tokens
    Returns list of clean tokens.
    """
    if not isinstance(text, str):
        return []

    text = text.lower()
    text = re.sub(r'http\S+|www\S+',          ' ', text)  # URLs
    text = re.sub(r'\S+@\S+',                 ' ', text)  # emails
    text = re.sub(r'\d+',                     ' ', text)  # numbers
    text = re.sub(r'[^a-z\s]',               ' ', text)  # punctuation
    text = re.sub(r'\s+',                     ' ', text).strip()

    tokens = word_tokenize(text)
    tokens = [t for t in tokens
              if t not in ALL_STOPWORDS and len(t) > 2]
    return tokens


# test it
sample = df['email_body'].iloc[0]
tokens = preprocess(sample)
print("Raw text (first 200 chars):")
print(sample[:200])
print(f"\nTokens ({len(tokens)} total):")
print(tokens[:30])

Raw text (first 200 chars):
Time is running very short. Is your company prepared for Etag 1.7? These are the minimum REQUIRED steps that must be completed prior to March 6, 2002:

---------------------------

(1) All entities mu

Tokens (88 total):
['time', 'running', 'short', 'prepared', 'etag', 'minimum', 'required', 'steps', 'must', 'completed', 'prior', 'march', 'entities', 'must', 'update', 'nerc', 'registry', 'must', 'designate', 'pors', 'pods', 'valid', 'tags', 'pse', 'must', 'designate', 'associated', 'scheduling', 'pors', 'pods']


In [5]:
print("Tokenizing email bodies...")
df['email_tokens'] = [preprocess(t) for t in tqdm(df['email_body'], desc="Emails")]

print("Tokenizing reply bodies...")
df['reply_tokens'] = [preprocess(t) for t in tqdm(df['reply_body'], desc="Replies")]

# combined corpus for Word2Vec training
all_sentences = df['email_tokens'].tolist() + df['reply_tokens'].tolist()

# stats
all_lengths = [len(t) for t in all_sentences]
print(f"\nTotal sentences for W2V  : {len(all_sentences):,}")
print(f"Avg tokens per sentence  : {np.mean(all_lengths):.1f}")
print(f"Median tokens            : {np.median(all_lengths):.1f}")
print(f"Empty token lists        : {sum(1 for t in all_sentences if len(t)==0):,}")

Tokenizing email bodies...


Emails: 100%|██████████| 48184/48184 [00:11<00:00, 4259.06it/s]


Tokenizing reply bodies...


Replies: 100%|██████████| 48184/48184 [00:10<00:00, 4499.03it/s]


Total sentences for W2V  : 96,368
Avg tokens per sentence  : 40.0
Median tokens            : 21.0
Empty token lists        : 267


In [6]:
print(f"Training Word2Vec  (dim={W2V_DIM}, window={W2V_WINDOW}, epochs={W2V_EPOCHS})...")
print("Expected time: ~1–2 min\n")

w2v_model = Word2Vec(
    sentences  = all_sentences,
    vector_size= W2V_DIM,
    window     = W2V_WINDOW,
    min_count  = 3,          # ignore words appearing < 3 times
    workers    = 4,
    epochs     = W2V_EPOCHS,
    sg         = 1,          # Skip-gram (better for rare words)
)

vocab_size = len(w2v_model.wv)
print(f"✅ Vocabulary size  : {vocab_size:,} words")
print(f"   Vector dim       : {W2V_DIM}")

# quick sanity check — similar words to common email words
test_words = ['meeting', 'price', 'report', 'gas', 'contract']
print("\n--- Word2Vec similarity checks ---")
for word in test_words:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=4)
        sim_str  = ", ".join([f"{w}({s:.2f})" for w, s in similar])
        print(f"  '{word}' → {sim_str}")
    else:
        print(f"  '{word}' → not in vocabulary")

# save
w2v_path = os.path.join(OUTPUT_DIR, "word2vec.model")
w2v_model.save(w2v_path)
print(f"\n✅ Word2Vec saved → {w2v_path}")

Training Word2Vec  (dim=128, window=5, epochs=10)...
Expected time: ~1–2 min

✅ Vocabulary size  : 33,152 words
   Vector dim       : 128

--- Word2Vec similarity checks ---
  'meeting' → meetings(0.77), ocotober(0.65), tuesday(0.64), rrg(0.63)
  'price' → payed(0.68), costless(0.68), prices(0.66), arithmetic(0.64)
  'report' → reports(0.69), surveyed(0.54), propt(0.53), gastrading(0.53)
  'gas' → natural(0.78), diverting(0.61), egpfc(0.60), nominates(0.59)
  'contract' → contracts(0.68), unexercised(0.62), fgu(0.61), editted(0.60)

✅ Word2Vec saved → embeddings\word2vec.model


In [7]:
def tokens_to_vec(tokens: list, model: Word2Vec) -> np.ndarray:
    """
    Mean-pool Word2Vec vectors for a list of tokens.
    Returns zero vector if no tokens found in vocabulary.
    """
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if not vecs:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

print("Building W2V sentence vectors...")
email_w2v = np.array([tokens_to_vec(t, w2v_model) 
                       for t in tqdm(df['email_tokens'], desc="Email W2V")])
reply_w2v = np.array([tokens_to_vec(t, w2v_model) 
                       for t in tqdm(df['reply_tokens'], desc="Reply W2V")])

print(f"\nemail_w2v shape : {email_w2v.shape}")
print(f"reply_w2v shape : {reply_w2v.shape}")

# save
np.save(os.path.join(OUTPUT_DIR, "email_w2v.npy"), email_w2v)
np.save(os.path.join(OUTPUT_DIR, "reply_w2v.npy"), reply_w2v)
print("✅ W2V sentence vectors saved")

Building W2V sentence vectors...


Reply W2V: 100%|██████████| 48184/48184 [00:03<00:00, 14884.72it/s]


email_w2v shape : (48184, 128)
reply_w2v shape : (48184, 128)
✅ W2V sentence vectors saved


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading SBERT model on : {device.upper()}")
print(f"Model                  : {SBERT_MODEL}")

sbert = SentenceTransformer(SBERT_MODEL, device=device)

# batch_size 128 works well on RTX 4060 (8GB VRAM)
BATCH = 128

print("\nEncoding email bodies...")
email_sbert = sbert.encode(
    df['email_body'].tolist(),
    batch_size       = BATCH,
    show_progress_bar= True,
    convert_to_numpy = True,
    normalize_embeddings = True   # L2 norm — makes cosine sim = dot product
)

print("\nEncoding reply bodies...")
reply_sbert = sbert.encode(
    df['reply_body'].tolist(),
    batch_size       = BATCH,
    show_progress_bar= True,
    convert_to_numpy = True,
    normalize_embeddings = True
)

print(f"\nemail_sbert shape : {email_sbert.shape}")
print(f"reply_sbert shape : {reply_sbert.shape}")

# save
np.save(os.path.join(OUTPUT_DIR, "email_sbert.npy"), email_sbert)
np.save(os.path.join(OUTPUT_DIR, "reply_sbert.npy"), reply_sbert)
print("✅ SBERT embeddings saved")

W0406 19:48:11.495000 27980 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading SBERT model on : CUDA
Model                  : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 700.65it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Encoding email bodies...


Batches: 100%|██████████| 377/377 [00:39<00:00,  9.49it/s]



Encoding reply bodies...


Batches: 100%|██████████| 377/377 [00:36<00:00, 10.37it/s]



email_sbert shape : (48184, 384)
reply_sbert shape : (48184, 384)
✅ SBERT embeddings saved


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

# sample 1000 pairs
idx = np.random.choice(len(df), size=min(1000, len(df)), replace=False)

# cosine sim between paired email↔reply (should be > random)
paired_sims = np.sum(email_sbert[idx] * reply_sbert[idx], axis=1)

# cosine sim between random shuffled pairs (baseline)
shuffled_idx = np.random.permutation(idx)
random_sims  = np.sum(email_sbert[idx] * reply_sbert[shuffled_idx], axis=1)

print("=" * 50)
print("   SBERT Embedding Sanity Check")
print("=" * 50)
print(f"  Paired  email↔reply cosine sim  : {paired_sims.mean():.4f} ± {paired_sims.std():.4f}")
print(f"  Random  email↔reply cosine sim  : {random_sims.mean():.4f} ± {random_sims.std():.4f}")
print(f"  Signal-to-noise ratio           : {paired_sims.mean()/random_sims.mean():.2f}x")
print("=" * 50)

if paired_sims.mean() > random_sims.mean():
    print("\n✅ Embeddings capture email-reply relationship!")
else:
    print("\n⚠️  Paired sim not higher — check data quality")

   SBERT Embedding Sanity Check
  Paired  email↔reply cosine sim  : 0.4387 ± 0.2764
  Random  email↔reply cosine sim  : 0.1691 ± 0.1158
  Signal-to-noise ratio           : 2.59x

✅ Embeddings capture email-reply relationship!


In [10]:
# save df with tokens (needed for W2V bipartite graph in Part 3)
df.drop(columns=['email_tokens','reply_tokens']).to_csv(PAIRS_CSV, index=False)

token_data = {
    'email_tokens': df['email_tokens'].tolist(),
    'reply_tokens': df['reply_tokens'].tolist(),
}
with open(os.path.join(OUTPUT_DIR, "tokens.pkl"), "wb") as f:
    pickle.dump(token_data, f)

print("=" * 50)
print("   PART 2 COMPLETE — Files saved")
print("=" * 50)
print(f"  embeddings/word2vec.model")
print(f"  embeddings/email_w2v.npy   → {email_w2v.shape}")
print(f"  embeddings/reply_w2v.npy   → {reply_w2v.shape}")
print(f"  embeddings/email_sbert.npy → {email_sbert.shape}")
print(f"  embeddings/reply_sbert.npy → {reply_sbert.shape}")
print(f"  embeddings/tokens.pkl")
print("=" * 50)

   PART 2 COMPLETE — Files saved
  embeddings/word2vec.model
  embeddings/email_w2v.npy   → (48184, 128)
  embeddings/reply_w2v.npy   → (48184, 128)
  embeddings/email_sbert.npy → (48184, 384)
  embeddings/reply_sbert.npy → (48184, 384)
  embeddings/tokens.pkl
